# 第 01 章：环境配置与 DeepSeek 启航

根据讲义，本章你将掌握以下核心能力：
- Agent-First 理念
- `create_agent` 的使用
- `stream_mode` 的流分块字典解析

## 1. 统一模型声明 (`init_chat_model`)
配置环境变量，借势动态工厂架构连接底层大脑。

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 统一接入入口
llm = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("https://api.deepseek.com")
)
print("模型实例建立完成。")

模型实例建立完成。


## 2. 实例化智能体大总管 (`create_agent`)
为大语言模型打造并外挂一个独立执行能力的系统查询时钟。

In [2]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def get_system_time(query: str) -> str:
    """返回当前系统的具体时间。"""
    import datetime
    return f"北京时间：{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

agent = create_agent(
    model=llm,
    tools=[get_system_time],
)

## 3. 按字典包裹追源解析 (`stream_mode="messages"`)
不再混乱解包，精准提取需要节点的文本！

In [3]:
query_time = "我需要你告诉我，现在北京是几点？"
print(f"--- 场景 A 提问: {query_time} ---")

for part in agent.stream(
    {"messages": [("user", query_time)]}, 
    stream_mode="messages", 
    version="v2"
):
    if part["type"] == "messages":
        message, metadata = part["data"]
        # 源头拦截：我们只看推理模型的产出文字，忽略日志打印过程。
        if metadata.get("langgraph_node") == "model":
            if message.content:
                print(message.content, end="", flush=True)

--- 场景 A 提问: 我需要你告诉我，现在北京是几点？ ---
我来帮你查看当前北京的时间。现在是北京时间 **2026年4月9日 16:00:29**（下午4点）。

In [4]:
query_general = "我现在不需要知道时间，请你简单和我打个招呼。"
print(f"--- 场景 B 提问: {query_general} ---")

for part in agent.stream(
    {"messages": [("user", query_general)]}, 
    stream_mode="messages", 
    version="v2"
):
    if part["type"] == "messages":
        message, metadata = part["data"]
        if metadata.get("langgraph_node") == "model":
            if message.content:
                print(message.content, end="", flush=True)

--- 场景 B 提问: 我现在不需要知道时间，请你简单和我打个招呼。 ---
你好！很高兴见到你！我是DeepSeek，一个乐于助人的AI助手。有什么我可以帮你的吗？